# ResolveX — 01: Train + Test Inference (run on KAGGLE — CPU notebook)

**Use the CPU notebook, not GPU.** The slow parts here (blocking, string
similarity features) are pandas/regex/rapidfuzz work that a GPU cannot
accelerate — LightGBM training itself (the only GPU-capable step) was never the
bottleneck. Kaggle's GPU notebooks give you *fewer* CPU cores (2) than the CPU
notebook (4), so GPU would make this slower, not faster. Pick accelerator =
**None** when creating the notebook.

## Before running

1. Zip your local `src` folder (the one containing `resolvex/`) and upload it as
   a new **private Kaggle Dataset**, e.g. named `resolvex-src`.
2. Zip your local `resources/dataset` folder (containing `train/` and `test/`)
   and upload it as another private Kaggle Dataset, e.g. named `resolvex-data`.
   This is still only the competition's own provided data — you're just moving
   *where* it's processed, not adding any external information, so this doesn't
   violate the "no external data" rule.
3. In this notebook's right-hand panel, **Add Input** → attach both datasets.
4. Settings → Accelerator → **None**. Session length: as long as you need (CPU
   sessions get ~12h on Kaggle vs ~9h for GPU/TPU, another reason to skip GPU).
5. Run all cells. At the end, download `/kaggle/working` (Kaggle zips it for you
   via the "Output" tab) — see the last cell for exactly what to bring back to
   your local project folder.


In [ ]:
import sys, os, glob

# Locate the uploaded resolvex package (handles either `src` or `src/resolvex`
# having been uploaded directly) and put its parent on sys.path.
pkg_matches = glob.glob("/kaggle/input/*/resolvex") + glob.glob("/kaggle/input/*/src/resolvex")
assert pkg_matches, (
    "Couldn't find the resolvex package under /kaggle/input. "
    "Upload your local `src` folder as a Kaggle dataset and attach it as input first."
)
sys.path.insert(0, pkg_matches[0].rsplit("/resolvex", 1)[0])

# Locate the uploaded dataset (must contain train/ and test/ subfolders).
data_matches = glob.glob("/kaggle/input/*/train/train_source1.tsv")
assert data_matches, (
    "Couldn't find train_source1.tsv under /kaggle/input. "
    "Upload your local resources/dataset folder as a Kaggle dataset and attach it as input first."
)
data_root = os.path.dirname(os.path.dirname(data_matches[0]))
print("resolvex package found, data root:", data_root)

os.environ["RESOLVEX_DATA_ROOT"] = data_root
os.environ["RESOLVEX_OUTPUT_ROOT"] = "/kaggle/working/output"
os.environ["RESOLVEX_EXPERIMENTS_ROOT"] = "/kaggle/working/experiments"


In [ ]:
!pip install -q lightgbm rapidfuzz pyarrow joblib


In [ ]:
import json
import time

import joblib
import numpy as np
import pandas as pd

from resolvex.blocking import generate_candidates
from resolvex.config import (
    CONFIGS_DIR, MODELS_DIR, METRICS_DIR, ERROR_ANALYSIS_DIR,
    TRAIN_GROUND_TRUTH, TRAIN_SOURCE1, TRAIN_SOURCE2, TRAIN_SOURCE3,
    TEST_SOURCE1, TEST_SOURCE2, TEST_SOURCE3,
)
from resolvex.data_loader import ground_truth_to_sets, load_ground_truth, load_source
from resolvex.evaluation import macro_f_beta
from resolvex.features import FEATURE_COLUMNS, compute_features_batched
from resolvex.models import ExactNameBaseline, RuleBasedBaseline, TfidfRetrievalBaseline, make_lgbm_model
from resolvex.pipeline import run_full_pipeline
from resolvex.postprocessing import write_tsv
from resolvex.thresholding import ThresholdConfig, apply_thresholds, optimize_threshold
from resolvex.training import fit_lgbm, grouped_train_val_split

for d in (CONFIGS_DIR, MODELS_DIR, METRICS_DIR, ERROR_ANALYSIS_DIR):
    d.mkdir(parents=True, exist_ok=True)


## Load training data and split (grouped by Source 1 entity — no leakage)

In [ ]:
print("Loading data...")
s1 = load_source(TRAIN_SOURCE1)
s2 = load_source(TRAIN_SOURCE2)
s3 = load_source(TRAIN_SOURCE3)
gt_sets_full = ground_truth_to_sets(load_ground_truth(TRAIN_GROUND_TRUTH))

s1_ids = s1["entity_id"].tolist()
s1_ids_set = set(s1_ids)
gt_sets = {k: v for k, v in gt_sets_full.items() if k in s1_ids_set}
train_ids, val_ids = grouped_train_val_split(s1_ids)
print(f"S1={len(s1):,} S2={len(s2):,} S3={len(s3):,} train={len(train_ids):,} val={len(val_ids):,}")


## Blocking (candidate generation) on the training set

In [ ]:
t0 = time.time()
candidates = generate_candidates(s1, s2, s3)
print(f"blocking: {time.time()-t0:.1f}s -> {len(candidates):,} pairs")


In [ ]:
def measure_blocking_quality(cands, gt_sets, s1_ids_set):
    cand_map = cands.groupby("source1_entity_id")["candidate_entity_id"].apply(set).to_dict()
    hit = total = zero_cand = with_truth = 0
    for s1_id in s1_ids_set:
        got = cand_map.get(s1_id, set())
        if not got:
            zero_cand += 1
        true_ids = gt_sets.get(s1_id, set())
        if not true_ids:
            continue
        with_truth += 1
        total += len(true_ids)
        hit += len(true_ids & got)
    n_per_entity = cands.groupby("source1_entity_id").size()
    return {
        "n_source1_entities": len(s1_ids_set),
        "entities_with_ground_truth_matches": with_truth,
        "true_matches_total": total,
        "true_matches_recovered": hit,
        "candidate_recall": hit / total if total else None,
        "zero_candidate_entities": zero_cand,
        "avg_candidates_per_entity": float(n_per_entity.mean()) if len(n_per_entity) else 0.0,
        "max_candidates_per_entity": int(n_per_entity.max()) if len(n_per_entity) else 0,
        "total_candidate_pairs": len(cands),
    }

quality = measure_blocking_quality(candidates, gt_sets, s1_ids_set)
quality["reduction_ratio"] = 1 - (len(candidates) / (len(s1) * (len(s2) + len(s3))))
print(quality)
with open(METRICS_DIR / "blocking_quality_full.json", "w") as f:
    json.dump(quality, f, indent=2)


## Label candidates (train/val split + positive/negative label) — vectorized

In [ ]:
candidates["split"] = np.where(candidates["source1_entity_id"].isin(train_ids), "train", "val")
gt_pairs_df = pd.DataFrame(
    [(s1id, m) for s1id, ms in gt_sets.items() for m in ms],
    columns=["source1_entity_id", "candidate_entity_id"],
)
gt_pairs_df["label"] = np.int8(1)
candidates = candidates.merge(gt_pairs_df, on=["source1_entity_id", "candidate_entity_id"], how="left")
candidates["label"] = candidates["label"].fillna(0).astype(np.int8)
print(candidates["label"].value_counts())


## Feature engineering

Uses `compute_features_batched` (chunks of 200k pairs at a time) to keep peak
memory bounded — at full scale candidates can run into the tens of millions of
rows, and computing features for all of them in one shot can exhaust RAM.

In [ ]:
is_s2 = candidates["candidate_entity_id"].str.startswith("S2-")
feat_parts = []
for mask, other_df in ((is_s2, s2), (~is_s2, s3)):
    subset = candidates[mask]
    if subset.empty:
        continue
    feats = compute_features_batched(subset, s1, other_df)
    feats["split"] = subset["split"].values
    feats["label"] = subset["label"].values
    feats["source1_entity_id"] = subset["source1_entity_id"].values
    feats["candidate_entity_id"] = subset["candidate_entity_id"].values
    feat_parts.append(feats)
all_feats = pd.concat(feat_parts, ignore_index=True)

train_mask = all_feats["split"] == "train"
val_mask = all_feats["split"] == "val"
X_train, y_train = all_feats.loc[train_mask, FEATURE_COLUMNS], all_feats.loc[train_mask, "label"]
X_val, y_val = all_feats.loc[val_mask, FEATURE_COLUMNS], all_feats.loc[val_mask, "label"]
print(f"train pairs={len(X_train):,} (pos={y_train.sum():,}) val pairs={len(X_val):,} (pos={y_val.sum():,})")


## Baselines (for comparison against the trained model)

In [ ]:
val_keys = all_feats.loc[val_mask, ["source1_entity_id", "candidate_entity_id"]].reset_index(drop=True)
val_feats_only = all_feats.loc[val_mask, FEATURE_COLUMNS].reset_index(drop=True)
val_s1_ids = [i for i in s1_ids if i in val_ids]
val_truth = {k: v for k, v in gt_sets.items() if k in val_ids}

baselines = {
    "exact_name": ExactNameBaseline(require_country=False),
    "exact_name_country": ExactNameBaseline(require_country=True),
    "weighted_fuzzy": RuleBasedBaseline(),
    "tfidf_retrieval": TfidfRetrievalBaseline(),
}
baseline_results = {}
for name, model in baselines.items():
    scored = val_keys.copy()
    scored["score"] = model.score(val_feats_only)
    best_cfg, grid = optimize_threshold(scored, val_truth, val_s1_ids)
    best_f = grid["macro_f0.5"].max()
    baseline_results[name] = {"best_threshold": best_cfg.accept_threshold, "macro_f0.5": float(best_f)}
    print(f"{name}: threshold={best_cfg.accept_threshold} macro_F0.5={best_f:.4f}")

with open(METRICS_DIR / "baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)


## Train LightGBM

In [ ]:
model = make_lgbm_model()
fit_lgbm(model, X_train, y_train, X_val, y_val)

val_scored = val_keys.copy()
val_scored["score"] = model.predict_proba(val_feats_only)[:, 1]


## Tune decision threshold on validation macro F_0.5 (never on test labels)

In [ ]:
best_cfg, grid = optimize_threshold(val_scored, val_truth, val_s1_ids)
print(f"Best threshold={best_cfg.accept_threshold} macro F0.5={grid['macro_f0.5'].max():.4f}")

joblib.dump(model, MODELS_DIR / "lgbm_matcher.joblib")
with open(CONFIGS_DIR / "threshold.json", "w") as f:
    json.dump(
        {"accept_threshold": best_cfg.accept_threshold, "singleton_abstain_threshold": best_cfg.singleton_abstain_threshold},
        f, indent=2,
    )
grid.to_csv(METRICS_DIR / "threshold_grid.csv", index=False)

final_metrics = macro_f_beta(apply_thresholds(val_scored, best_cfg), val_truth, entity_ids=val_s1_ids)
with open(METRICS_DIR / "val_metrics.json", "w") as f:
    json.dump(final_metrics, f, indent=2)
print("Validation metrics:", final_metrics)


## Error analysis: sample false positives / false negatives on validation

In [ ]:
predictions = apply_thresholds(val_scored, best_cfg)
name_lookup = pd.concat([s1, s2, s3]).set_index("entity_id")["business_name"].to_dict()
addr_lookup = pd.concat([s1, s2, s3]).set_index("entity_id")["business_address"].to_dict()

fps, fns = [], []
for s1_id in val_s1_ids:
    pred = predictions.get(s1_id, set())
    truth = val_truth.get(s1_id, set())
    for fp in pred - truth:
        fps.append((s1_id, fp))
    for fn in truth - pred:
        fns.append((s1_id, fn))

def describe(pairs, limit=30):
    return [
        {
            "source1_entity_id": a, "other_entity_id": b,
            "s1_name": name_lookup.get(a, ""), "s1_address": addr_lookup.get(a, ""),
            "other_name": name_lookup.get(b, ""), "other_address": addr_lookup.get(b, ""),
        }
        for a, b in pairs[:limit]
    ]

report = {
    "n_false_positives": len(fps), "n_false_negatives": len(fns),
    "false_positive_examples": describe(fps), "false_negative_examples": describe(fns),
}
with open(ERROR_ANALYSIS_DIR / "error_examples.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(f"FPs={len(fps)} FNs={len(fns)}")


---
## Test-set inference

Everything above only touched training data. This runs the full pipeline
(blocking → features → scoring → thresholding → postprocessing) on the test
set and writes the two competition output files.

In [ ]:
print("Loading test data...")
test_s1 = load_source(TEST_SOURCE1)
test_s2 = load_source(TEST_SOURCE2)
test_s3 = load_source(TEST_SOURCE3)
print(f"test S1={len(test_s1):,} S2={len(test_s2):,} S3={len(test_s3):,}")

t0 = time.time()
result = run_full_pipeline(test_s1, test_s2, test_s3, model, best_cfg)
print(f"test pipeline: {time.time()-t0:.1f}s")

from resolvex.config import MATCHING_RESULTS_PATH, CANDIDATE_PAIRS_PATH, OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_tsv(result["matching_results"], MATCHING_RESULTS_PATH)
write_tsv(result["candidate_pairs"], CANDIDATE_PAIRS_PATH)
print(f"Wrote {MATCHING_RESULTS_PATH}")
print(f"Wrote {CANDIDATE_PAIRS_PATH}")


## Done — what to bring back to your local project

Download `/kaggle/working` (Output tab → Download All). It contains:

```
output/matching_results.tsv
output/candidate_pairs.tsv
experiments/models/lgbm_matcher.joblib
experiments/configs/threshold.json
experiments/metrics/blocking_quality_full.json
experiments/metrics/baseline_results.json
experiments/metrics/threshold_grid.csv
experiments/metrics/val_metrics.json
experiments/error_analysis/error_examples.json
```

Copy each of these into the **same relative path** in your local `ResolveX`
folder (e.g. `output/matching_results.tsv` → `D:\Projects\ResolveX\output\matching_results.tsv`).
Then run `notebooks/02_validate_and_package_local.ipynb` locally to validate and
zip the submission.